# COME 03 — Inference: folder of images, single frame, and the dashboard server

Uses the research U-Net checkpoint from notebook 01 and the EfficientNet-B0 classifier.
Cloud cover is computed **inside the sky disk** (an overcast sky gives 8 oktas), each image is segmented itself,
and no random augmentation is applied at prediction time.


In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from torchvision import models, transforms

UNET_CKPT  = Path("come_runs/run_YYYYMMDD_HHMMSS/unet_sessionpure.pt")   # from notebook 01
CLS_CKPT   = Path("model/efficientnet_finetuned2.pth")
IMAGES     = Path("/home/champion/Documents/.Codes/Python/AMB82-Mini/videos/img_extraction/days/2026")
OUT        = IMAGES / "predictions"; (OUT / "masks").mkdir(parents=True, exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
"""Image geometry and the Red-Blue Difference Index (RBDI).

Identical to the preprocessing used for the reported results:
square centre crop -> 480x480 -> circular sky disk (radius 480/2 - 23 px) -> RBDI normalised over the disk.
All functions are numpy/OpenCV only.
"""
import cv2
import numpy as np

HARM = 480          # harmonised side length (pixels)
DISK_MARGIN = 23    # pixels trimmed from the disk radius to exclude the lens rim
MODEL_SIZE = 320    # network input side length


def sky_disk(size=HARM, margin=DISK_MARGIN):
    """Boolean mask of the circular sky disk."""
    yy, xx = np.mgrid[0:size, 0:size]
    r = size // 2 - margin
    return ((xx - size / 2) ** 2 + (yy - size / 2) ** 2) <= r * r


def harmonise(img, msk=None, size=HARM):
    """Square centre crop and resize to `size`; returns (img, msk, disk)."""
    h, w = img.shape[:2]
    s = min(h, w)
    oy, ox = (h - s) // 2, (w - s) // 2
    img = cv2.resize(img[oy:oy + s, ox:ox + s], (size, size), interpolation=cv2.INTER_AREA)
    if msk is not None:
        msk = cv2.resize(msk[oy:oy + s, ox:ox + s], (size, size), interpolation=cv2.INTER_NEAREST)
    return img, msk, sky_disk(size)


def rbdi(img_bgr, disk):
    """Return (float image in [0,1], RBDI = (R-B) min-max normalised over the sky disk)."""
    f = img_bgr.astype(np.float32) / 255.0
    rb = f[..., 2] - f[..., 0]
    lo, hi = rb[disk].min(), rb[disk].max()
    return f, (rb - lo) / max(1e-6, hi - lo)


def model_input(img_bgr, msk=None, model_size=MODEL_SIZE):
    """Build the 4-channel (B, G, R, RBDI) network input exactly as in training.

    Returns x (4, S, S) float32, y (1, S, S) float32 or None, d (1, S, S) bool sky disk.
    """
    img, msk, disk = harmonise(img_bgr, msk)
    f, I = rbdi(img, disk)
    x480 = np.stack([f[..., 0], f[..., 1], f[..., 2], I], -1).astype(np.float32)
    x = cv2.resize(x480, (model_size, model_size), interpolation=cv2.INTER_AREA).transpose(2, 0, 1).copy()
    d = cv2.resize(disk.astype(np.uint8) * 255, (model_size, model_size), interpolation=cv2.INTER_NEAREST) > 127
    y = None
    if msk is not None:
        y = (cv2.resize(msk, (model_size, model_size), interpolation=cv2.INTER_NEAREST) > 127).astype(np.float32)[None]
    return x, y, d[None]


def solid_angle_weights(size=HARM, margin=DISK_MARGIN):
    """Per-pixel solid-angle weights for an equidistant fisheye (zero outside the disk)."""
    disk = sky_disk(size, margin)
    yy, xx = np.mgrid[0:size, 0:size]
    r = size // 2 - margin
    rr = np.sqrt((xx - size / 2) ** 2 + (yy - size / 2) ** 2)
    k = (np.pi / 2) / r
    w = k * np.sin(k * np.clip(rr, 0, r)) / np.clip(rr, 1e-6, r)
    w[~disk] = 0.0
    return w


In [ ]:
"""Cloud fraction and oktas, computed INSIDE the sky disk only.

Counting the black corners outside the fisheye circle as clear sky would cap the result at
about pi/4 * 8 = 6.3 oktas, so the denominator here is always the number of sky-disk pixels.
"""
import math
import numpy as np


def cloud_fraction(cloud_mask, disk, weights=None):
    cloud = (np.asarray(cloud_mask) > 0) & disk
    if weights is None:
        return float(cloud.sum() / max(1, disk.sum()))
    return float((cloud * weights).sum() / max(1e-12, (disk * weights).sum()))


def to_okta(fraction):
    """floor(8 * fraction), capped at 8 (the convention used throughout the study)."""
    return int(min(8, math.floor(8 * fraction + 1e-9)))


In [ ]:
"""U-Net used for all reported segmentation results.

Input: 4 channels (B, G, R, RBDI) at 320x320. Base width 32 (32-64-128-256 encoder, 512 bottleneck),
two 3x3 conv + BatchNorm + ReLU per block, transposed-conv upsampling. About 7.77 M parameters.
"""
import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True),
                               nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True))

    def forward(self, x):
        return self.b(x)


class UNet(nn.Module):
    def __init__(self, in_ch=4, c=32):
        super().__init__()
        self.p = nn.MaxPool2d(2)
        self.e1, self.e2 = DoubleConv(in_ch, c), DoubleConv(c, 2 * c)
        self.e3, self.e4 = DoubleConv(2 * c, 4 * c), DoubleConv(4 * c, 8 * c)
        self.b = DoubleConv(8 * c, 16 * c)
        self.u3, self.d3 = nn.ConvTranspose2d(16 * c, 8 * c, 2, 2), DoubleConv(16 * c, 8 * c)
        self.u2, self.d2 = nn.ConvTranspose2d(8 * c, 4 * c, 2, 2), DoubleConv(8 * c, 4 * c)
        self.u1, self.d1 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2), DoubleConv(4 * c, 2 * c)
        self.u0, self.d0 = nn.ConvTranspose2d(2 * c, c, 2, 2), DoubleConv(2 * c, c)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.p(x1)); x3 = self.e3(self.p(x2)); x4 = self.e4(self.p(x3))
        y = self.d3(torch.cat([self.u3(self.b(self.p(x4))), x4], 1))
        y = self.d2(torch.cat([self.u2(y), x3], 1))
        y = self.d1(torch.cat([self.u1(y), x2], 1))
        return self.out(self.d0(torch.cat([self.u0(y), x1], 1)))


In [ ]:
CLASS_NAMES = ["A-clear sky", "B-pattern (Altocumulus)", "C-thick-dark (Nimbostratus)", "D-thick-white (Cumulus)", "E-veil (Cirrus)"]
CLS_TF = transforms.Compose([transforms.ToPILImage(), transforms.Resize((224, 224)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])   # deterministic

seg = UNet().to(dev); st = torch.load(UNET_CKPT, map_location=dev)
seg.load_state_dict(st.get("model_state_dict", st)); seg.eval()
cls = models.efficientnet_b0(weights=None); cls.classifier[1] = nn.Linear(cls.classifier[1].in_features, len(CLASS_NAMES))
cls.load_state_dict(torch.load(CLS_CKPT, map_location=dev)); cls.to(dev).eval()

@torch.no_grad()
def analyse(img_bgr, threshold=0.5):
    x, _, d = model_input(img_bgr)
    prob = seg(torch.from_numpy(x)[None].to(dev)).sigmoid()[0, 0].cpu().numpy()
    mask = (prob > threshold) & d[0]
    f = cloud_fraction(mask, d[0])
    p = torch.softmax(cls(CLS_TF(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))[None].to(dev)), 1)[0].cpu().numpy()
    return dict(mask=(mask * 255).astype(np.uint8), cloud_fraction=f, okta=to_okta(f),
                cloud_type=CLASS_NAMES[int(p.argmax())], probabilities=dict(zip(CLASS_NAMES, map(float, p))))


In [ ]:
rows = []
for p in sorted(x for x in IMAGES.iterdir() if x.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}):
    img = cv2.imread(str(p))
    if img is None: print("unreadable:", p.name); continue
    r = analyse(img)
    cv2.imwrite(str(OUT / "masks" / f"{p.stem}_mask.png"), r["mask"])
    rows.append(dict(image=p.name, cloud_fraction=round(r["cloud_fraction"], 4), okta=r["okta"], cloud_type=r["cloud_type"]))
pred = pd.DataFrame(rows); pred.to_csv(OUT / "predictions.csv", index=False); pred


## Dashboard server
The next cell writes `come_server.py` next to this notebook; the cell after starts it on port 5000 (stop it with the Jupyter stop button).


In [ ]:
%%writefile come_server.py
"""HTTP server for the dashboard: POST raw JPEG bytes to /upload_frame and /get_cloud_types."""
import os, json
from datetime import datetime
from pathlib import Path
import cv2, numpy as np
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import come_infer as ci                     # written by the cell below

app = FastAPI()
SAVE = Path(os.environ.get("COME_SAVE_DIR", "captures"))

def _decode(b): return cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_COLOR)

@app.post("/upload_frame")
async def upload_frame(request: Request):
    img = _decode(await request.body())
    if img is None: return JSONResponse({"error": "could not decode image"}, status_code=400)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f"); day = SAVE / stamp[:8]; day.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(day / f"{stamp}.jpg"), img)
    r = ci.analyse(img); cv2.imwrite(str(day / f"{stamp}_mask.png"), r["mask"])
    return {"timestamp": stamp, "cloud_fraction": r["cloud_fraction"], "cloud_coverage": r["okta"]}

@app.post("/get_cloud_types")
async def get_cloud_types(request: Request):
    img = _decode(await request.body())
    if img is None: return JSONResponse({"error": "could not decode image"}, status_code=400)
    r = ci.analyse(img); return {"cloud_type": r["cloud_type"], "probabilities": r["probabilities"]}

@app.get("/get_time")
async def get_time():
    n = datetime.now(); return {"date": n.date().isoformat(), "time": n.time().isoformat(timespec="seconds")}


In [ ]:
%%writefile come_infer.py
"""Image geometry and the Red-Blue Difference Index (RBDI).

Identical to the preprocessing used for the reported results:
square centre crop -> 480x480 -> circular sky disk (radius 480/2 - 23 px) -> RBDI normalised over the disk.
All functions are numpy/OpenCV only.
"""
import cv2
import numpy as np

HARM = 480          # harmonised side length (pixels)
DISK_MARGIN = 23    # pixels trimmed from the disk radius to exclude the lens rim
MODEL_SIZE = 320    # network input side length


def sky_disk(size=HARM, margin=DISK_MARGIN):
    """Boolean mask of the circular sky disk."""
    yy, xx = np.mgrid[0:size, 0:size]
    r = size // 2 - margin
    return ((xx - size / 2) ** 2 + (yy - size / 2) ** 2) <= r * r


def harmonise(img, msk=None, size=HARM):
    """Square centre crop and resize to `size`; returns (img, msk, disk)."""
    h, w = img.shape[:2]
    s = min(h, w)
    oy, ox = (h - s) // 2, (w - s) // 2
    img = cv2.resize(img[oy:oy + s, ox:ox + s], (size, size), interpolation=cv2.INTER_AREA)
    if msk is not None:
        msk = cv2.resize(msk[oy:oy + s, ox:ox + s], (size, size), interpolation=cv2.INTER_NEAREST)
    return img, msk, sky_disk(size)


def rbdi(img_bgr, disk):
    """Return (float image in [0,1], RBDI = (R-B) min-max normalised over the sky disk)."""
    f = img_bgr.astype(np.float32) / 255.0
    rb = f[..., 2] - f[..., 0]
    lo, hi = rb[disk].min(), rb[disk].max()
    return f, (rb - lo) / max(1e-6, hi - lo)


def model_input(img_bgr, msk=None, model_size=MODEL_SIZE):
    """Build the 4-channel (B, G, R, RBDI) network input exactly as in training.

    Returns x (4, S, S) float32, y (1, S, S) float32 or None, d (1, S, S) bool sky disk.
    """
    img, msk, disk = harmonise(img_bgr, msk)
    f, I = rbdi(img, disk)
    x480 = np.stack([f[..., 0], f[..., 1], f[..., 2], I], -1).astype(np.float32)
    x = cv2.resize(x480, (model_size, model_size), interpolation=cv2.INTER_AREA).transpose(2, 0, 1).copy()
    d = cv2.resize(disk.astype(np.uint8) * 255, (model_size, model_size), interpolation=cv2.INTER_NEAREST) > 127
    y = None
    if msk is not None:
        y = (cv2.resize(msk, (model_size, model_size), interpolation=cv2.INTER_NEAREST) > 127).astype(np.float32)[None]
    return x, y, d[None]


def solid_angle_weights(size=HARM, margin=DISK_MARGIN):
    """Per-pixel solid-angle weights for an equidistant fisheye (zero outside the disk)."""
    disk = sky_disk(size, margin)
    yy, xx = np.mgrid[0:size, 0:size]
    r = size // 2 - margin
    rr = np.sqrt((xx - size / 2) ** 2 + (yy - size / 2) ** 2)
    k = (np.pi / 2) / r
    w = k * np.sin(k * np.clip(rr, 0, r)) / np.clip(rr, 1e-6, r)
    w[~disk] = 0.0
    return w


"""Cloud fraction and oktas, computed INSIDE the sky disk only.

Counting the black corners outside the fisheye circle as clear sky would cap the result at
about pi/4 * 8 = 6.3 oktas, so the denominator here is always the number of sky-disk pixels.
"""
import math
import numpy as np


def cloud_fraction(cloud_mask, disk, weights=None):
    cloud = (np.asarray(cloud_mask) > 0) & disk
    if weights is None:
        return float(cloud.sum() / max(1, disk.sum()))
    return float((cloud * weights).sum() / max(1e-12, (disk * weights).sum()))


def to_okta(fraction):
    """floor(8 * fraction), capped at 8 (the convention used throughout the study)."""
    return int(min(8, math.floor(8 * fraction + 1e-9)))


"""U-Net used for all reported segmentation results.

Input: 4 channels (B, G, R, RBDI) at 320x320. Base width 32 (32-64-128-256 encoder, 512 bottleneck),
two 3x3 conv + BatchNorm + ReLU per block, transposed-conv upsampling. About 7.77 M parameters.
"""
import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True),
                               nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True))

    def forward(self, x):
        return self.b(x)


class UNet(nn.Module):
    def __init__(self, in_ch=4, c=32):
        super().__init__()
        self.p = nn.MaxPool2d(2)
        self.e1, self.e2 = DoubleConv(in_ch, c), DoubleConv(c, 2 * c)
        self.e3, self.e4 = DoubleConv(2 * c, 4 * c), DoubleConv(4 * c, 8 * c)
        self.b = DoubleConv(8 * c, 16 * c)
        self.u3, self.d3 = nn.ConvTranspose2d(16 * c, 8 * c, 2, 2), DoubleConv(16 * c, 8 * c)
        self.u2, self.d2 = nn.ConvTranspose2d(8 * c, 4 * c, 2, 2), DoubleConv(8 * c, 4 * c)
        self.u1, self.d1 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2), DoubleConv(4 * c, 2 * c)
        self.u0, self.d0 = nn.ConvTranspose2d(2 * c, c, 2, 2), DoubleConv(2 * c, c)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        x1 = self.e1(x); x2 = self.e2(self.p(x1)); x3 = self.e3(self.p(x2)); x4 = self.e4(self.p(x3))
        y = self.d3(torch.cat([self.u3(self.b(self.p(x4))), x4], 1))
        y = self.d2(torch.cat([self.u2(y), x3], 1))
        y = self.d1(torch.cat([self.u1(y), x2], 1))
        return self.out(self.d0(torch.cat([self.u0(y), x1], 1)))


import os
from pathlib import Path
import cv2, numpy as np, torch, torch.nn as nn
from torchvision import models, transforms
UNET_CKPT = Path(os.environ['COME_UNET']); CLS_CKPT = Path(os.environ['COME_CLS']); dev = 'cpu'


CLASS_NAMES = ["A-clear sky", "B-pattern (Altocumulus)", "C-thick-dark (Nimbostratus)", "D-thick-white (Cumulus)", "E-veil (Cirrus)"]
CLS_TF = transforms.Compose([transforms.ToPILImage(), transforms.Resize((224, 224)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])   # deterministic

seg = UNet().to(dev); st = torch.load(UNET_CKPT, map_location=dev)
seg.load_state_dict(st.get("model_state_dict", st)); seg.eval()
cls = models.efficientnet_b0(weights=None); cls.classifier[1] = nn.Linear(cls.classifier[1].in_features, len(CLASS_NAMES))
cls.load_state_dict(torch.load(CLS_CKPT, map_location=dev)); cls.to(dev).eval()

@torch.no_grad()
def analyse(img_bgr, threshold=0.5):
    x, _, d = model_input(img_bgr)
    prob = seg(torch.from_numpy(x)[None].to(dev)).sigmoid()[0, 0].cpu().numpy()
    mask = (prob > threshold) & d[0]
    f = cloud_fraction(mask, d[0])
    p = torch.softmax(cls(CLS_TF(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))[None].to(dev)), 1)[0].cpu().numpy()
    return dict(mask=(mask * 255).astype(np.uint8), cloud_fraction=f, okta=to_okta(f),
                cloud_type=CLASS_NAMES[int(p.argmax())], probabilities=dict(zip(CLASS_NAMES, map(float, p))))


In [ ]:
import os, subprocess, sys
env = dict(os.environ, COME_UNET=str(UNET_CKPT.resolve()), COME_CLS=str(CLS_CKPT.resolve()))
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "come_server:app", "--host", "0.0.0.0", "--port", "5000"], env=env)
print("server running on port 5000 (pid", server.pid, ") - run server.terminate() to stop it")
